<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_98_generalization_bayesian_dl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 📊 **Module 98 — Generalization Theory + Bayesian Deep Learning** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 14 · Theory & Responsibility (UDL deep-dives)


# Module 98 — Generalization Theory + Bayesian Deep Learning

> Classical ML statisticians said over-parameterized networks should **overfit catastrophically**. Modern deep networks have **billions of parameters** trained on millions of samples and generalize fine — sometimes *better* the more parameters you add. This module is the canon explaining *why*, built around UDL **Chapter 8** (Performance: bias-variance, **double descent**, high-dim spaces) + **Chapter 9** (Regularization: L2, implicit, ensembling, **Bayesian**, augmentation) + **Chapter 20** (Generalization: random data, **lottery tickets**, adversarial) — eight notebooks total. Modernized to 2026 with **deep ensembles · MC Dropout · Laplace approximation · stochastic weight averaging · conformal prediction · calibration**.

### What you'll cover
1. The **bias-variance trade-off** — classical view
2. **Double descent** — why over-parameterization can *help*
3. **High-dimensional geometry** — what changes in 1000 D
4. **L2 / weight decay / implicit regularization** — same animal, three angles
5. **Lottery Ticket Hypothesis** (Frankle 2018) — sparse subnetworks
6. **Bayesian deep learning** — why we want a posterior, not a point estimate
7. **MC Dropout · Deep Ensembles · SWAG · Laplace approximation** — practical Bayesian
8. **Calibration** — temperature scaling, isotonic, Platt
9. **Conformal prediction** — distribution-free coverage guarantees
10. The **2026 production stack** — when uncertainty matters and how to deploy it


## 1 · The classical bias-variance trade-off

Decompose expected test error of an estimator `f̂` at point `x`:

```
E[(y − f̂(x))²] = bias² + variance + noise
                = (E[f̂(x)] − f(x))²  +  E[(f̂(x) − E[f̂(x)])²]  +  σ²
```

- **Bias** — wrong functional form (linear model on cubic data)
- **Variance** — sensitivity to which training set you happened to draw
- **Noise** — irreducible (you can't predict a coin flip)

Classical sweet-spot picture (the textbook **U-curve**):

```
test error
   │  bias dominates  ◄── too simple
   │\
   │ \
   │  \                  ◄── sweet spot
   │   \__/\__
   │        |
   │        |     ◄── too complex / variance dominates
   │
   └──────────────► model capacity
```

This *exactly* describes pre-2014 ML — fit a degree-3 polynomial to a degree-3 process, get the best test error; degrees 5+ overfit. **And then deep learning broke it.**

> 📐 **The bias-variance equation is correct; the *capacity axis* changed.** Classical ML was on the *under-parameterized* side: more parameters → less bias, more variance. Deep nets sit on the *over-parameterized* side where interpolation begins and a second descent kicks in.


## 2 · Double descent

**Belkin, Hsu, Ma, Mandal (PNAS 2019)** formalized what practitioners had seen for years: as you grow model capacity past the **interpolation threshold** (parameters ≈ training data), test error *first rises, then falls again*.

```
test error
   │   classical U-curve         interpolation       MODERN MONOTONIC
   │     |                            ▼                      ▼
   │\   ▲                            /\
   │ \  | sweet                     /  \          ___________
   │  \_| spot                     /    \________/
   │      \                        |
   │       \______________________/
   │                              ↑
   │                       parameters ≈ data (the threshold)
   └──────────────────────────────────────────────► capacity
```

**Two regimes:**
- **Under-parameterized** — classical bias-variance; sweet spot is interior
- **Over-parameterized** — *every* model interpolates training data; among those, the network's **implicit regularization** picks the *flattest minimum*, which generalizes best

**Why over-parameterized helps.** With many more parameters than data, infinitely many solutions interpolate the training set. SGD doesn't pick a random one — it picks the **minimum-norm** (L2-norm) solution implicitly. Flatter, lower-norm solutions generalize.

> 🌟 **Practical implication.** Don't try to find the bottom of the classical U-curve. Push past the interpolation threshold; you'll often get *better* test error than the classical sweet spot — for free.


## 3 · High-dimensional geometry — what UDL Chap 8.4 shows

UDL's `8_4_High_Dimensional_Spaces.ipynb` walks through facts that are genuinely surprising and underlie a lot of deep-learning behavior:

| Fact | What it means |
|---|---|
| **Volume concentrates in the shell** | In high D, almost all of a uniform-ball's volume sits near its surface |
| **Random vectors are nearly orthogonal** | Two random unit vectors in 1000 D have inner product `≈ N(0, 1/1000)` — essentially perpendicular |
| **Distances concentrate** | All pairs of i.i.d. samples have ≈ the same distance; "nearest neighbor" becomes weak |
| **Linear separators almost always exist** | N points in d ≫ N dimensions are linearly separable with high probability (Cover's theorem) |
| **The blessing-of-dimensionality** | Adversarial perturbations exist (M91) but so do many "good" directions to find |

These are the geometric facts that make **over-parameterized networks generalize** and make **gradient methods find good minima** despite the apparent non-convexity. In 1000 D, almost every direction at a saddle point has positive curvature; you can almost always escape.


## 4 · L2 / weight decay / implicit regularization

Three formally distinct things often used interchangeably:

| Name | Form | Where |
|---|---|---|
| **L2 regularization** | add `λ·||w||²` to the loss | SGD as gradient descent on loss + λ||w||² |
| **Weight decay** | multiplicative `w ← (1 − λ·lr)·w` per step | AdamW (Loshchilov 2019); not equivalent to L2 for adaptive optimizers |
| **Implicit regularization** | SGD itself biases toward low-norm / flat solutions | Free — comes from the optimizer, not the loss |

**Why AdamW exists.** Loshchilov 2019 showed adding `λ||w||²` to the loss in Adam doesn't actually shrink large weights — Adam's per-coordinate scaling cancels it. AdamW *explicitly* decays weights after the update step. **Every 2026 production-grade Adam optimizer uses the W variant.**

**Implicit regularization (Soudry 2018, Gunasekar 2018).** For separable data, gradient descent on logistic loss with no explicit regularization converges to the **max-margin** classifier. The optimizer is implicitly an SVM. Extends to deep linear networks (Arora 2019): GD prefers low-rank solutions when the architecture is deep.

> 🎯 **What this means for tuning**. You're regularizing whether or not you put `λ` in your loss. **Weight decay 0.01-0.1** + AdamW is the universal 2026 default; explicit L2 is the textbook version most modern code does *not* use.


## 5 · Lottery Ticket Hypothesis — UDL Chap 20.3

**Frankle & Carbin (ICLR 2019, best paper)**: inside any randomly initialized over-parameterized network there exists a **sparse subnetwork** ("winning ticket") that, when trained from the same initial weights, **matches or exceeds the dense network's accuracy**.

```
1. Initialize a random dense network θ_0
2. Train to completion → θ_T
3. Prune the smallest |θ_T| weights → mask m
4. RESET surviving weights to their ORIGINAL θ_0 values (same init!)
5. Train again with mask m → matches θ_T
6. Iteratively prune → "winning ticket" can be 1-5% of original
```

The initialization and the mask together encode the trainable subnetwork. **Rewinding** (Frankle 2020) showed even better results: instead of resetting to step 0, reset to step k (a few epochs in) — the early training has positioned the network better than random init.

**Implications:**
- **Sparse training** is possible at the same accuracy if you can find the ticket cheaply (RigL, SET, Top-KAST)
- **Continual pretraining** can preserve / reactivate tickets across tasks
- The lottery view explains why **edge models** (M90) work: ship the ticket, not the dense network

> 🪙 **The intellectual move.** Lottery Tickets reframed pruning from "remove redundancy after training" to "discover the architecture that was always there." This view directly motivates **structured-sparsity training** (2:4 sparsity, NVIDIA Ampere+) and **mixture-of-experts** (MoE — activate a sub-network per token).


## 6 · Bayesian deep learning — why we want a posterior

Point-estimate networks output `ŷ = f_θ(x)`. Bayesian networks output a **distribution** `p(y | x, D)` over possible predictions:

```
p(y | x, D) = ∫ p(y | x, θ) · p(θ | D) dθ
```

The posterior `p(θ | D)` is the catch — exact for tiny models, intractable for deep nets. Different approximations gave us the modern Bayesian DL canon:

| Method | Idea | Cost |
|---|---|---|
| **MC Dropout** (Gal 2016) | Dropout at *test* time too; `T` stochastic forward passes ≈ posterior samples | T× one model |
| **Deep Ensembles** (Lakshminarayanan 2017) | Train `K` independent models; uncertainty = disagreement | K× train + K× inference |
| **Laplace approximation** (MacKay 1992 → Daxberger 2021) | Posterior ≈ Gaussian around MAP; uses the loss Hessian | one extra Hessian estimate |
| **SWAG** (Maddox 2019) | Stochastic Weight Averaging + low-rank Gaussian over SGD iterates | nearly free post-training |
| **SVI / Bayes by Backprop** (Blundell 2015) | Variational inference over weights | 2× memory, 2× compute |
| **HMC / NUTS / Langevin** | Markov chain over weights | expensive (research-grade) |

**Why we care about uncertainty:**
- **Safety-critical decisions** (medical triage, self-driving) — refuse to act when uncertain
- **Active learning** — label the points the model is most uncertain about
- **OOD detection** — uncertainty rises far from training distribution
- **Bayesian optimization** — exploration-vs-exploitation requires uncertainty
- **Reinforcement learning** — distributional value functions, exploration bonuses

The 2026 production picks: **Deep Ensembles** if you can afford K× compute; **MC Dropout** as the cheap fallback; **Laplace** for one-shot post-hoc; **SWAG** when you've already trained.


## 7 · Practical Bayesian DL — code-level

**MC Dropout.** Add `nn.Dropout(p)` between layers (already common). At test time, do NOT call `model.eval()` for the dropout layers — keep them stochastic. Average T forward passes:

```python
def mc_predict(model, x, T=30):
    model.train()                          # dropout on
    preds = torch.stack([model(x) for _ in range(T)])
    return preds.mean(0), preds.var(0)     # mean prediction, predictive variance
```

**Deep Ensembles.** Train K models with different random seeds (different init + different data shuffles). Predict with the average; uncertainty is variance across the ensemble. K=5 captures ~80% of the benefit; K=10 captures ~95%.

**Laplace approximation (Daxberger 2021 `laplace-torch`).** After training the MAP estimate `θ̂`, approximate the posterior by `N(θ̂, H⁻¹)` where `H` is the Hessian of the loss. The Kronecker-factored approximation (KFAC) makes it tractable for large networks. **Post-hoc** — costs ~one extra epoch.

**SWAG (Stochastic Weight Averaging — Gaussian).** During the last few epochs of SGD, accumulate `θ̄ = mean(θ_k)` and a low-rank covariance over the iterates `θ_k`. Predict by sampling from `N(θ̄, Σ)`. Nearly free; runner-up to deep ensembles in practice.

> 🧪 **Empirical leaderboard for uncertainty quality** (UCI benchmarks, computer-vision OOD): **Deep Ensembles ≈ HMC > SWAG > Laplace ≈ MC Dropout > SVI > single MAP**.


## 8 · Calibration — Brier · ECE · temperature scaling

A model is **calibrated** when its predicted probability ≈ its empirical accuracy. A perfectly calibrated 80% prediction is correct 80% of the time.

| Metric | Formula | Interpretation |
|---|---|---|
| **Brier score** | `(1/N) Σ (p_i − y_i)²` | Mean squared error of probabilities |
| **Expected Calibration Error (ECE)** | bin predictions by confidence, compute mean |bin accuracy − bin confidence| | The textbook calibration metric |
| **Reliability diagram** | bar chart: per-confidence-bin accuracy | Visual check |
| **NLL (negative log-likelihood)** | `−(1/N) Σ log p(y_i | x_i)` | Penalizes overconfidence |

**Calibration fixes (post-hoc, hold-out a calibration set):**

| Method | Idea |
|---|---|
| **Temperature scaling** (Guo 2017) | One scalar `T`: `softmax(logits / T)`; fit on held-out NLL |
| **Vector / matrix scaling** | Per-class temperatures |
| **Platt scaling** | Logistic regression on logits |
| **Isotonic regression** | Non-parametric monotonic fit |
| **Beta calibration** | Beta-distribution fit; better than Platt for skewed |

**Why deep nets are miscalibrated.** Modern networks with cross-entropy loss + weight decay are **overconfident** — the softmax saturates at the wrong side of the decision boundary. **Temperature scaling (one scalar) fixes ~80% of the gap** with no architectural change. Always run it before shipping.

> 🌡️ **Temperature scaling in 10 lines:** hold out 5% of train, optimize one `T` to minimize NLL on it, divide logits by `T` at inference. The simplest, most-effective post-hoc fix in deep learning.


In [ ]:
import torch, torch.nn.functional as F

def temperature_scale(logits, labels):
    T = torch.nn.Parameter(torch.ones(1))
    opt = torch.optim.LBFGS([T], lr=0.01, max_iter=50)
    def step():
        opt.zero_grad()
        loss = F.cross_entropy(logits / T, labels)
        loss.backward(); return loss
    opt.step(step)
    return T.item()                            # use this T at inference


## 9 · Conformal prediction — distribution-free guarantees

Frequentist counter to Bayesian DL. **Conformal prediction** (Vovk 2005 → Angelopoulos 2021 "Gentle Introduction") gives finite-sample, distribution-free coverage:

> *For any classifier `f̂`, any user-chosen miscoverage `α ∈ (0,1)`, and any held-out calibration set, conformal prediction produces a prediction set `C(x)` that contains the true `y` with probability **at least 1−α**.*

**Split conformal in 4 lines** (the dominant 2026 form):

```
1. Train any model on D_train
2. Compute "nonconformity score" s_i = 1 − p̂_{y_i}(x_i) for each (x_i, y_i) ∈ D_cal
3. Threshold q = ⌈(N+1)(1−α)⌉-th smallest of {s_i}
4. Predict C(x) = {y : 1 − p̂_y(x) ≤ q}
```

That set is **provably 1−α covering** under exchangeability — no Gaussian assumption, no calibration tuning, no model retraining. **Adaptive Prediction Sets (APS, Romano 2020)** and **RAPS (Angelopoulos 2021)** make the sets *also* size-efficient.

| Use | Why conformal helps |
|---|---|
| **Medical diagnosis** | "These 3 conditions cover the patient's true label with 95% confidence" |
| **Drug design** | Sets of plausible candidate molecules |
| **Autonomous driving** | Distributional risk-aware planning |
| **Active learning** | Larger sets = label these points first |
| **LLM uncertainty** | Conformal Set over generations (Quach 2024) |

**Conformal is the rigor frequentists wanted** in deep learning, and the **regulatory-friendly** way to ship uncertainty under EU AI Act / FDA requirements (M100 callback).

> 🛡️ **What conformal cannot do.** Distribution shift breaks the exchangeability assumption. Modern work (Tibshirani 2019 "weighted CP", Gibbs 2022 "online CP") extends conformal to covariate shift and online streaming.


## 10 · The 2026 production stack — when uncertainty matters

| Library | Use |
|---|---|
| **`laplace-torch`** (Daxberger) | Post-hoc Laplace; KFAC, full-Hessian, last-layer |
| **`torchuq`** (Wen) | One-stop uncertainty: ensembles, dropout, temperature, ECE |
| **`Pyro` / `NumPyro`** | Full BNN posteriors via HMC, SVI |
| **`uncertainty-baselines`** (Google) | Reference implementations of every method above |
| **`mapie`** (sklearn-style) | Production conformal prediction |
| **`crepes`** | Conformal in Python; tabular + structured |
| **`torchcp`** | Modern conformal for PyTorch + LLMs |

**Production decision tree:**

```
                Do you need uncertainty at all?
                          │
        ┌─────────────────┴────────────────────┐
       NO                                    YES
   (single MAP)                                │
                          ┌───────────────────┴─────────────────────┐
                  Need calibration only?               Need predictive distribution?
                          │                                   │
                  Temperature scaling                ┌────────┴─────────┐
                  (5 lines, 80% of gap)        Cheapest?            Best?
                                                MC Dropout          Deep Ensemble (K=5-10)
                                                + Laplace           + SWAG late-training
                                                  │
                                            Need rigorous coverage?
                                            → Conformal prediction
```

> 🎓 **The takeaway.** A 2026 production ML system has *three* uncertainty layers: (a) **calibrated probabilities** via temperature scaling, (b) **predictive distribution** via deep ensemble or MC dropout, (c) **rigorous coverage** via conformal. Most teams ship (a). Sophisticated teams ship (a)+(c). Safety-critical teams (medical, autonomous, scientific) ship all three. This is the modern responsible ML stack.

## ✅ Recap

- **Bias-variance** is still the right decomposition — but the capacity axis grew, and **double descent** says over-parameterized often beats the classical sweet spot.
- **High-dim geometry** (concentration of measure, orthogonality, separability) underwrites why DL generalizes.
- **L2 ≠ weight decay** for adaptive optimizers — **AdamW** is the modern fix.
- **Implicit regularization**: SGD prefers low-norm / flat / max-margin solutions for free.
- **Lottery Ticket Hypothesis** — sparse subnetworks exist inside dense networks at full accuracy; motivates 2:4 sparsity, MoE, edge AI.
- **Bayesian DL** approximates the posterior via **MC Dropout · Deep Ensembles · SWAG · Laplace · SVI**. Deep Ensembles ≈ HMC for quality, K=5 captures 80% of the gain.
- **Calibration**: temperature scaling fixes ~80% of overconfidence in 5 lines — always do this.
- **Conformal prediction**: distribution-free finite-sample coverage. The regulatory-grade uncertainty story.
- **2026 stack**: `laplace-torch · torchuq · uncertainty-baselines · mapie · torchcp`.

Next: **M99 — Optimization Deep Dive**.
